# Group-based re-split (fixes patient/session leakage) + YOLO box validation (removes 27 invalid boxes) + second cross-dataset deduplication against the segmenter (20 exact matches removed). Generates Dataset Tratado 2, consumed by Tratado_3 and by Tratado_0's downstream check.

Resolve the two critical issues identified by the co-advisor (Correc1):

1. **Information leakage between partitions** (same patient/session in
   train and val) → re-partitioning by group (acquisition identifier).
2. **Invalid YOLO boxes** (e.g., relative area 2.58779) → thorough validation
   of every line in each `.txt` file, detailed log, and exclusion of
   unrecoverable data.

Finally, it generates:
- The partition evidence table (identifiers per split + 0
  matches) requested by the co-advisor.
- The recalculated area statistics (mean, median, min, valid max,
  Q1, Q3, 95th percentile, number of invalid entries, % <1%).
- The corrected dataset in a folder ready for use in the notebook for
  merging with negatives.

## 0. Configuration and Imports


In [ ]:
import os
import re
import json
import shutil
from pathlib import Path
from collections import defaultdict, Counter

import cv2
import numpy as np
import pandas as pd

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
BASE = Path("/content/drive/MyDrive/UNIVERSIDAD/9NO SEMESTRE/TITULACION/DATASET CARIES")
IMAGES_ROOT = BASE / "Dataset Tratado 1" / "images"
LABELS_ROOT = BASE / "Dataset Tratado 1" / "labels"

OUTPUT_ROOT = BASE / "Dataset Tratado 2"
if OUTPUT_ROOT.exists():
    shutil.rmtree(OUTPUT_ROOT)
for split in ["train", "val", "test"]:
    (OUTPUT_ROOT / "images" / split).mkdir(parents=True, exist_ok=True)
    (OUTPUT_ROOT / "labels" / split).mkdir(parents=True, exist_ok=True)

SPLIT_RATIOS = {"train": 0.70, "val": 0.15, "test": 0.15}
SEED = 42

CLASS_NAMES = ["Caries"]

assert IMAGES_ROOT.exists() and LABELS_ROOT.exists(), "No se encontraron images/ o labels/ en BASE"
print("Dataset base:", BASE)

Dataset base: /content/drive/MyDrive/UNIVERSIDAD/9NO SEMESTRE/TITULACION/DATASET CARIES


## 1. Current inventory (all partitions combined)

The train, val, and test partitions are recombined into a single pool, because the current partition
is precisely the one that needs to be redone.

In [ ]:
registros = []
for split_actual in ["train", "val", "test"]:
    img_dir = IMAGES_ROOT / split_actual
    lbl_dir = LABELS_ROOT / split_actual
    if not img_dir.exists():
        continue
    for img_p in sorted(img_dir.iterdir()):
        if img_p.suffix.lower() not in (".jpg", ".jpeg", ".png"):
            continue
        lbl_p = lbl_dir / f"{img_p.stem}.txt"
        registros.append({
            "stem": img_p.stem,
            "image_path": img_p,
            "label_path": lbl_p if lbl_p.exists() else None,
            "split_original": split_actual,
        })

print(f"Total de imágenes en el pool combinado: {len(registros)}")

Total de imágenes en el pool combinado: 1645


### CELL 1.1 — Exclude images detected during cross-validation

In [ ]:
import pandas as pd

RUTA_EXCLUSION = BASE / "imagenes_excluidas_por_fuga.csv"
df_excluir = pd.read_csv(RUTA_EXCLUSION)
archivos_a_excluir = set(df_excluir["archivo_a_excluir"])

n_antes = len(registros)
registros = [r for r in registros if r["image_path"].name not in archivos_a_excluir]
n_despues = len(registros)

print(f"Registros antes de excluir por fuga: {n_antes}")
print(f"Registros después de excluir por fuga: {n_despues}")
print(f"Excluidos: {n_antes - n_despues}")


Registros antes de excluir por fuga: 1645
Registros después de excluir por fuga: 1465
Excluidos: 180


## 2. Extraction of the patient/session identifier

Pattern observed in the file names (e.g., `003-008-621-00`,
`003_007_316_00`): three blocks of digits separated by a hyphen or underscore,
followed by a final two-digit block. That block is captured
as the group identifier, ignoring the view (Frontal, Mandibular,
etc.) and the timestamp, which may vary between images of the same
patient/session.

In [ ]:
PATRON_ID = re.compile(r"(\d{3}[-_]\d{3}[-_]\d+[-_]\d{2})")

def extraer_grupo(stem: str) -> str:
    m = PATRON_ID.search(stem)
    if m:
        return re.sub(r"[-_]", "-", m.group(1))  # normalizar separador
    return stem  # fallback: imagen huérfana, grupo = ella misma

for r in registros:
    r["grupo"] = extraer_grupo(r["stem"])

grupos_counter = Counter(r["grupo"] for r in registros)
sin_patron = sum(1 for r in registros if r["grupo"] == r["stem"])

print(f"Grupos únicos detectados: {len(grupos_counter)}")
print(f"Imágenes sin patrón reconocido (grupo = ellas mismas): {sin_patron}")
print("\nEjemplo de grupos con más de una imagen (deberían ir juntas en el mismo split):")
for g, n in grupos_counter.most_common(10):
    if n > 1:
        print(f"  {g}: {n} imágenes")

Grupos únicos detectados: 1062
Imágenes sin patrón reconocido (grupo = ellas mismas): 209

Ejemplo de grupos con más de una imagen (deberían ir juntas en el mismo split):
  003-008-1188-00: 5 imágenes
  003-008-620-01: 5 imágenes
  003-008-1052-00: 4 imágenes
  003-008-1134-00: 4 imágenes
  003-008-1140-00: 4 imágenes
  003-008-621-00: 4 imágenes
  003-008-625-00: 4 imágenes
  003-008-816-00: 4 imágenes
  003-103-348-00: 4 imágenes
  003-007-423-00: 4 imágenes


## 3. Redistribution by group (70/15/15)

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

idx_all = np.arange(len(registros))
grupos_arr = np.array([r["grupo"] for r in registros])

gss1 = GroupShuffleSplit(n_splits=1, train_size=SPLIT_RATIOS["train"], random_state=SEED)
train_idx, temp_idx = next(gss1.split(idx_all, groups=grupos_arr))

val_frac_de_temp = SPLIT_RATIOS["val"] / (SPLIT_RATIOS["val"] + SPLIT_RATIOS["test"])
gss2 = GroupShuffleSplit(n_splits=1, train_size=val_frac_de_temp, random_state=SEED)
temp_grupos_arr = grupos_arr[temp_idx]
val_idx_rel, test_idx_rel = next(gss2.split(temp_idx, groups=temp_grupos_arr))
val_idx = temp_idx[val_idx_rel]
test_idx = temp_idx[test_idx_rel]

for i in train_idx:
    registros[i]["split_nuevo"] = "train"
for i in val_idx:
    registros[i]["split_nuevo"] = "val"
for i in test_idx:
    registros[i]["split_nuevo"] = "test"

print("Nueva distribución por split:")
for s in ["train", "val", "test"]:
    n = sum(1 for r in registros if r["split_nuevo"] == s)
    print(f"  {s}: {n} imágenes")

Nueva distribución por split:
  train: 1028 imágenes
  val: 222 imágenes
  test: 215 imágenes


## 4. Partition Evidence Table (the one requested by the co-advisor)

In [ ]:
grupos_train = {r["grupo"] for r in registros if r["split_nuevo"] == "train"}
grupos_val = {r["grupo"] for r in registros if r["split_nuevo"] == "val"}
grupos_test = {r["grupo"] for r in registros if r["split_nuevo"] == "test"}

tabla_evidencia = pd.DataFrame({
    "Comprobación": [
        "Identificadores en entrenamiento",
        "Identificadores en validación",
        "Identificadores en prueba",
        "Coincidencias entrenamiento–validación",
        "Coincidencias entrenamiento–prueba",
        "Coincidencias validación–prueba",
    ],
    "Resultado": [
        len(grupos_train),
        len(grupos_val),
        len(grupos_test),
        len(grupos_train & grupos_val),
        len(grupos_train & grupos_test),
        len(grupos_val & grupos_test),
    ],
})

print(tabla_evidencia.to_string(index=False))
tabla_evidencia.to_csv(BASE / "tabla_evidencia_particion.csv", index=False)
print(f"\nGuardada en: {BASE / 'tabla_evidencia_particion.csv'}")

assert tabla_evidencia.loc[3:5, "Resultado"].sum() == 0, "⚠️ Aún hay coincidencias entre particiones — revisar el patrón de ID."
print("\n✅ Cero coincidencias entre particiones confirmado.")

                          Comprobación  Resultado
      Identificadores en entrenamiento        743
         Identificadores en validación        159
             Identificadores en prueba        160
Coincidencias entrenamiento–validación          0
    Coincidencias entrenamiento–prueba          0
       Coincidencias validación–prueba          0

Guardada en: /content/drive/MyDrive/UNIVERSIDAD/9NO SEMESTRE/TITULACION/DATASET CARIES/tabla_evidencia_particion.csv

✅ Cero coincidencias entre particiones confirmado.


## Markdown ready to paste into the article / response to the co-instructor

In [ ]:
md_tabla = tabla_evidencia.to_markdown(index=False)
print(md_tabla)

| Comprobación                           |   Resultado |
|:---------------------------------------|------------:|
| Identificadores en entrenamiento       |         743 |
| Identificadores en validación          |         159 |
| Identificadores en prueba              |         160 |
| Coincidencias entrenamiento–validación |           0 |
| Coincidencias entrenamiento–prueba     |           0 |
| Coincidencias validación–prueba        |           0 |


## 5. Validation of Each YOLO Line Against the 11 Correc1 Rules

Rules verified per line:
- exactly 5 fields;
- valid class identifier (0, single class);
- 0 ≤ xc ≤ 1;  0 ≤ yc ≤ 1;
- 0 < w ≤ 1;   0 < h ≤ 1;
- xc − w/2 ≥ 0;  xc + w/2 ≤ 1;
- yc − h/2 ≥ 0;  yc + h/2 ≤ 1;
- finite numerical values.

In [ ]:
def validar_linea_yolo(partes):
    """Devuelve (es_valida: bool, tipo_error: str|None)."""
    if len(partes) != 5:
        return False, "num_campos_invalido"
    try:
        cls_id = int(partes[0])
        xc, yc, w, h = (float(p) for p in partes[1:])
    except ValueError:
        return False, "valor_no_numerico"

    if not np.isfinite([xc, yc, w, h]).all():
        return False, "valor_no_finito"
    if cls_id != 0:
        return False, "id_clase_invalido"
    if not (0.0 <= xc <= 1.0):
        return False, "xc_fuera_de_rango"
    if not (0.0 <= yc <= 1.0):
        return False, "yc_fuera_de_rango"
    if not (0.0 < w <= 1.0):
        return False, "w_fuera_de_rango"
    if not (0.0 < h <= 1.0):
        return False, "h_fuera_de_rango"
    if xc - w / 2 < 0:
        return False, "caja_excede_borde_izquierdo"
    if xc + w / 2 > 1.0:
        return False, "caja_excede_borde_derecho"
    if yc - h / 2 < 0:
        return False, "caja_excede_borde_superior"
    if yc + h / 2 > 1.0:
        return False, "caja_excede_borde_inferior"
    return True, None

In [ ]:
lineas_invalidas = []  # log detallado para reportar a la cotutora
areas_validas = []

for r in registros:
    if r["label_path"] is None:
        r["lineas_validas"] = []
        continue

    lineas_ok = []
    with open(r["label_path"], "r") as f:
        for n_linea, linea in enumerate(f, start=1):
            linea_strip = linea.strip()
            if not linea_strip:
                continue
            partes = linea_strip.split()
            es_valida, tipo_error = validar_linea_yolo(partes)
            if es_valida:
                lineas_ok.append(linea_strip)
                w, h = float(partes[3]), float(partes[4])
                areas_validas.append(w * h)
            else:
                lineas_invalidas.append({
                    "particion_original": r["split_original"],
                    "split_nuevo": r["split_nuevo"],
                    "archivo": r["label_path"].name,
                    "linea": n_linea,
                    "valores_originales": linea_strip,
                    "tipo_error": tipo_error,
                })
    r["lineas_validas"] = lineas_ok

print(f"Total de líneas inválidas detectadas: {len(lineas_invalidas)}")
if lineas_invalidas:
    df_invalidas = pd.DataFrame(lineas_invalidas)
    print(df_invalidas.to_string(index=False))
    df_invalidas.to_csv(BASE / "cajas_invalidas_detectadas.csv", index=False)
    print(f"\nGuardado el detalle en: {BASE / 'cajas_invalidas_detectadas.csv'}")
    print("\n>>> Antes de excluirlas definitivamente: revisa la anotación original")
    print(">>> (LabelMe/COCO/PASCAL VOC) de cada caso listado arriba para intentar")
    print(">>> regenerar la conversión correctamente. Solo excluye lo que no sea recuperable.")
else:
    print("✅ No se detectaron cajas inválidas.")

Total de líneas inválidas detectadas: 27
particion_original split_nuevo                                                                                    archivo  linea                    valores_originales                  tipo_error
             train       train             no_retractors_Frontal_anonymous_003-007-1214-00_1732862917124_Frontal_View.txt      1 0 0.201286 1.298267 0.285706 0.596533           yc_fuera_de_rango
             train       train             no_retractors_Frontal_anonymous_003-008-1188-00_1732714947165_Frontal_View.txt      1 0 0.165401 1.012410 0.235020 0.024820           yc_fuera_de_rango
             train       train             no_retractors_Frontal_anonymous_003-008-1188-00_1732714947165_Frontal_View.txt      2 0 0.995074 1.184854 0.009852 0.369708           yc_fuera_de_rango
             train       train             no_retractors_Frontal_anonymous_003-008-1188-00_1732714947165_Frontal_View.txt      3 0 0.180634 1.155790 0.228492 0.311581           yc

## 6. Recalculated area statistics (valid boxes only)

In [ ]:
areas_validas = np.array(areas_validas)
n_invalidas = len(lineas_invalidas)

print("--- ESTADÍSTICAS DE ÁREA RELATIVA (recalculadas, solo cajas válidas) ---")
print(f"N cajas válidas: {len(areas_validas)}")
print(f"N cajas inválidas excluidas: {n_invalidas}")
print(f"Media: {areas_validas.mean():.5f}")
print(f"Mediana: {np.median(areas_validas):.5f}")
print(f"Mínimo: {areas_validas.min():.5f}")
print(f"Máximo válido: {areas_validas.max():.5f}")
print(f"Q1: {np.percentile(areas_validas, 25):.5f}")
print(f"Q3: {np.percentile(areas_validas, 75):.5f}")
print(f"Percentil 95: {np.percentile(areas_validas, 95):.5f}")
pct_pequenas = (areas_validas < 0.01).mean() * 100
print(f"% de cajas <1% del área de la imagen: {pct_pequenas:.1f}%")
print(f"% de cajas inválidas sobre el total original: {n_invalidas / (len(areas_validas) + n_invalidas):.2%}")

--- ESTADÍSTICAS DE ÁREA RELATIVA (recalculadas, solo cajas válidas) ---
N cajas válidas: 4622
N cajas inválidas excluidas: 27
Media: 0.03108
Mediana: 0.03113
Mínimo: 0.00006
Máximo válido: 0.09133
Q1: 0.02462
Q3: 0.03727
Percentil 95: 0.04574
% de cajas <1% del área de la imagen: 0.5%
% de cajas inválidas sobre el total original: 0.58%


## 7. Saving the corrected dataset (new partition + valid boxes only)

In [ ]:
for r in registros:
    split = r["split_nuevo"]
    out_img = OUTPUT_ROOT / "images" / split / r["image_path"].name
    out_lbl = OUTPUT_ROOT / "labels" / split / f"{r['stem']}.txt"

    shutil.copy(r["image_path"], out_img)
    with open(out_lbl, "w") as f:
        f.write("\n".join(r.get("lineas_validas", [])))
        if r.get("lineas_validas"):
            f.write("\n")

data_yaml_content = f"""train: images/train
val: images/val
test: images/test

nc: {len(CLASS_NAMES)}
names: {CLASS_NAMES}
"""
(OUTPUT_ROOT / "data.yaml").write_text(data_yaml_content)

print("Dataset corregido guardado en:", OUTPUT_ROOT)

Dataset corregido guardado en: /content/drive/MyDrive/UNIVERSIDAD/9NO SEMESTRE/TITULACION/DATASET CARIES/Dataset Tratado 2


### CELL 7.1 Cross-deduplication against the segmenter (SHA-256 + pHash)

In [ ]:
# Second, independent leakage check (post group re-split), restricted to val/test — see Tratado_0 for the first pass on the full pool.

!pip install imagehash --quiet

import hashlib
import imagehash
from PIL import Image

DATASET_DIENTES = Path("/content/drive/MyDrive/UNIVERSIDAD/9NO SEMESTRE/TITULACION/DATASET DIENTES")

def sha256_de(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()

def phash_de(path: Path):
    try:
        return imagehash.phash(Image.open(path))
    except Exception:
        return None

# --- 1) Hashear el lado del segmentador (solo train y valid: lo que el
#        modelo usó para aprender/validar; test del segmentador se deja
#        fuera porque no participó del entrenamiento) ---
registros_dientes = []
for split in ["train", "valid"]:
    for img_p in (DATASET_DIENTES / split / "images").glob("*"):
        registros_dientes.append({
            "path": img_p,
            "split_dientes": split,
            "sha256": sha256_de(img_p),
            "phash": phash_de(img_p),
        })

print(f"Imágenes del segmentador (train+valid) hasheadas: {len(registros_dientes)}")

# --- 2) Hashear el lado de caries: solo val y test (calibración y prueba
#        del pipeline; train de caries no importa para esta verificación) ---
registros_caries = []
for split in ["val", "test"]:
    img_dir = OUTPUT_ROOT / "images" / split
    for img_p in img_dir.glob("*"):
        registros_caries.append({
            "path": img_p,
            "split_caries": split,
            "sha256": sha256_de(img_p),
            "phash": phash_de(img_p),
        })

print(f"Imágenes de caries (val+test) hasheadas: {len(registros_caries)}")

# --- 3) Buscar coincidencias exactas (SHA-256) y perceptuales (pHash, distancia ≤ 5) ---
UMBRAL_HAMMING = 5
coincidencias = []

sha_dientes = {r["sha256"]: r for r in registros_dientes}

for rc in registros_caries:
    # Coincidencia exacta
    if rc["sha256"] in sha_dientes:
        rd = sha_dientes[rc["sha256"]]
        coincidencias.append({
            "imagen_caries": rc["path"].name, "split_afectado": rc["split_caries"],
            "imagen_dientes_coincidente": rd["path"].name, "split_dientes": rd["split_dientes"],
            "tipo_coincidencia": "sha256", "distancia_hamming": 0,
        })
        continue
    # Coincidencia perceptual
    if rc["phash"] is None:
        continue
    for rd in registros_dientes:
        if rd["phash"] is None:
            continue
        dist = rc["phash"] - rd["phash"]  # distancia de Hamming
        if dist <= UMBRAL_HAMMING:
            coincidencias.append({
                "imagen_caries": rc["path"].name, "split_afectado": rc["split_caries"],
                "imagen_dientes_coincidente": rd["path"].name, "split_dientes": rd["split_dientes"],
                "tipo_coincidencia": "phash", "distancia_hamming": int(dist),
            })
            break  # una coincidencia basta para marcar exclusión

df_dedup = pd.DataFrame(coincidencias)
print(f"\nCoincidencias encontradas: {len(df_dedup)}")
if len(df_dedup) > 0:
    print(df_dedup.to_string(index=False))

# --- 4) Excluir del lado de caries (val/test) las imágenes coincidentes ---
n_excluidas = 0
for _, fila in df_dedup.iterrows():
    split = fila["split_afectado"]
    nombre = fila["imagen_caries"]
    img_p = OUTPUT_ROOT / "images" / split / nombre
    lbl_p = OUTPUT_ROOT / "labels" / split / f"{Path(nombre).stem}.txt"
    if img_p.exists():
        img_p.unlink()
        n_excluidas += 1
    if lbl_p.exists():
        lbl_p.unlink()

df_dedup["accion"] = "excluida"
RUTA_TABLA_DEDUP = BASE / "tabla_deduplicacion_cruzada.csv"
df_dedup.to_csv(RUTA_TABLA_DEDUP, index=False)

print(f"\n✅ {n_excluidas} imágenes excluidas de val/test por coincidir con el segmentador.")
print(f"📄 Tabla guardada en: {RUTA_TABLA_DEDUP}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 6.9 MB/s eta 0:00:00
Imágenes del segmentador (train+valid) hasheadas: 1760
Imágenes de caries (val+test) hasheadas: 437

Coincidencias encontradas: 20
                                                                                    imagen_caries split_afectado                                                                                                 imagen_dientes_coincidente split_dientes tipo_coincidencia  distancia_hamming
                               pilot_Maxillary_Occlusal_anonymous-maxillaryView-1726650785522.jpg            val                                 c_Maxillary_Occlusal_anonymous-maxillaryView-1726650785522_jpg.rf.ZWlTegoKq5C5OfpRdkdi.jpg         valid            sha256                  0
                retractors_Mandibular_anonymous_003-007-1337-01_1733552404003_Mandibular_View.jpg            val                       c_Mandibular_anonymous_003-007-1337-01_1733552404003_Mandibular_View_jpg.rf.P1B8

## 8. Final Summary by Split (to be included in the article, Section 2.2.2)

In [ ]:
print("--- RESUMEN FINAL DEL DATASET CORREGIDO ---")
total_img, total_ann, total_neg = 0, 0, 0
for split in ["train", "val", "test"]:
    imgs = list((OUTPUT_ROOT / "images" / split).glob("*"))
    n_img = len(imgs)
    n_ann = 0
    n_neg = 0
    for img_p in imgs:
        lbl_p = OUTPUT_ROOT / "labels" / split / f"{img_p.stem}.txt"
        lineas = [l for l in open(lbl_p) if l.strip()] if lbl_p.exists() else []
        n_ann += len(lineas)
        if len(lineas) == 0:
            n_neg += 1
    print(f"{split.upper()}: {n_img} imágenes, {n_ann} cajas, {n_neg} negativas")
    total_img += n_img
    total_ann += n_ann
    total_neg += n_neg

print(f"\nTOTAL: {total_img} imágenes, {total_ann} cajas, {total_neg} negativas")
print("\n>>> Usa estos números como CARIES_ROOT_CORREGIDO en el notebook de fusión con negativos.")

--- RESUMEN FINAL DEL DATASET CORREGIDO ---
TRAIN: 1028 imágenes, 3189 cajas, 31 negativas
VAL: 211 imágenes, 687 cajas, 3 negativas
TEST: 206 imágenes, 693 cajas, 2 negativas

TOTAL: 1445 imágenes, 4569 cajas, 36 negativas

>>> Usa estos números como CARIES_ROOT_CORREGIDO en el notebook de fusión con negativos.
